# 05. Дизайн A/B-теста: расчёт размера выборки

В ноутбуке 03 я рекомендую запустить триггерную кампанию реактивации для сегмента с низким первым чеком и проверить эффект через A/B-тест. Здесь надо посчитать, сколько клиентов потребуется в каждой группе, чтобы поймать прирост retention на 2 п.п. с приличной мощностью.

Делаем по шагам:
1. Считаем базовую конверсию (что считать H0).
2. Формулируем H1: базовая конверсия плюс минимально интересный эффект.
3. Считаем размер выборки по формуле для двух пропорций.
4. Строим power curve: как требуемый n меняется с размером эффекта.
5. Проводим Монте-Карло симуляцию: запускаем тысячи виртуальных тестов и смотрим, как часто наш дизайн ловит реальный эффект.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
rng = np.random.default_rng(42)

customers = pd.read_parquet('../data/customers_labeled.parquet')
print(f'Клиентов в данных: {len(customers):,}')

## Базовая конверсия в сегменте

Конверсией здесь считаю долю клиентов, у которых после первой покупки случилась хотя бы вторая. Берём её именно по сегменту 'низкий первый чек', потому что тестировать кампанию мы планируем на нём.

In [ ]:
low = customers[customers['check_segment'].str.startswith('Низкий')]
p_baseline = low['returned'].mean()
print(f'Размер сегмента: {len(low):,} клиентов')
print(f'Базовая доля возврата (p_baseline): {p_baseline*100:.2f}%')

## Параметры теста

Фиксируем заранее, чтобы не подгонять задним числом:
- Уровень значимости alpha = 0.05 (двусторонний тест, потому что эффект может быть и отрицательным).
- Желаемая мощность 1 - beta = 0.80. Это стандарт в продуктовой аналитике.
- MDE (минимально детектируемый эффект) = 2 п.п. в абсолюте. Меньше нет смысла ловить, потому что разница в 1 п.п. не окупит стоимость самой кампании.

In [ ]:
ALPHA = 0.05
POWER = 0.80
MDE = 0.02  # абсолютные 2 п.п.

p_test = p_baseline + MDE
print(f'p_baseline (контроль):  {p_baseline*100:.2f}%')
print(f'p_test     (тест):      {p_test*100:.2f}%')
print(f'абсолютный MDE:         {MDE*100:.1f} п.п.')
print(f'относительный лифт:     {MDE/p_baseline*100:.1f}%')

## Считаем размер выборки

Используем стандартный аппарат из statsmodels: проверяем гипотезу о равенстве двух пропорций. Под капотом простая формула с z-квантилями для alpha и beta, но удобнее не выводить вручную, а взять готовое.

In [ ]:
effect_size = proportion_effectsize(p_test, p_baseline)
analysis = NormalIndPower()
n_per_group = analysis.solve_power(
    effect_size=effect_size,
    alpha=ALPHA,
    power=POWER,
    ratio=1.0,
    alternative='two-sided',
)
n_per_group = int(np.ceil(n_per_group))
n_total = n_per_group * 2

print(f'Effect size (Cohen h):       {effect_size:.4f}')
print(f'Нужно на одну группу:        {n_per_group:,} клиентов')
print(f'Всего на тест:               {n_total:,} клиентов')
print(f'Размер сегмента в данных:    {len(low):,} клиентов')
print()
print(f'Хватит ли нам данных для одного тестового цикла? {"Да" if len(low) >= n_total else "Нет"}')

Если у нас в реальности есть только определённый поток новых низкочековых клиентов в месяц, можно прикинуть, сколько недель будет крутиться тест. Это уже разговор не про статистику, а про продуктовый менеджмент.

## Power curve: как меняется требуемый n с размером эффекта

Полезный график для разговора с продактом: 'если мы хотим ловить эффект 1 п.п. вместо 2 п.п., тест станет в N раз длиннее'.

In [ ]:
mde_grid = np.linspace(0.005, 0.05, 30)
n_required = []
for mde in mde_grid:
    es = proportion_effectsize(p_baseline + mde, p_baseline)
    n = analysis.solve_power(effect_size=es, alpha=ALPHA, power=POWER, ratio=1.0, alternative='two-sided')
    n_required.append(int(np.ceil(n)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mde_grid * 100, n_required, color='#4C72B0', linewidth=2)
ax.axvline(x=MDE * 100, color='red', linestyle='--', alpha=0.6, label=f'наш MDE = {MDE*100:.0f} п.п.')
ax.set_yscale('log')
ax.set_xlabel('MDE, абсолютные процентные пункты')
ax.set_ylabel('Размер одной группы (лог-шкала)')
ax.set_title('Сколько нужно клиентов на одну группу для разных MDE')
ax.legend()
plt.tight_layout()
plt.savefig('../images/power_curve.png', dpi=120, bbox_inches='tight')
plt.show()

Важный практический момент: график лог-шкальный, и это не случайно. Между MDE = 1% и MDE = 2% разница в стоимости теста (по числу пользователей) не двукратная, а четырёхкратная. Поэтому продуктовая команда обычно соглашается на MDE = 2 п.п. как на компромисс между чувствительностью и длительностью теста.

## Монте-Карло симуляция: проверяем, что наш дизайн действительно работает

Формула это хорошо, но интуитивно убедительнее увидеть симуляцию. Возьмём наш расчётный размер выборки, прогоним 2000 виртуальных A/B-тестов с известным истинным эффектом и посмотрим, в каком проценте тестов мы поймали значимый результат. Если получится около 80%, значит расчёт мощности корректный.

In [ ]:
N_SIMULATIONS = 2000
p_control = p_baseline
p_treatment = p_baseline + MDE

significant_results = 0
for _ in range(N_SIMULATIONS):
    control_outcomes = rng.binomial(1, p_control, size=n_per_group)
    treat_outcomes = rng.binomial(1, p_treatment, size=n_per_group)
    # двухвыборочный z-тест на разницу пропорций через scipy
    n1, n2 = n_per_group, n_per_group
    p1 = control_outcomes.mean()
    p2 = treat_outcomes.mean()
    p_pool = (control_outcomes.sum() + treat_outcomes.sum()) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
    z = (p2 - p1) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    if p_value < ALPHA:
        significant_results += 1

empirical_power = significant_results / N_SIMULATIONS
print(f'Запущено симуляций:           {N_SIMULATIONS:,}')
print(f'Из них значимых результатов:  {significant_results:,}')
print(f'Эмпирическая мощность:        {empirical_power*100:.1f}%')
print(f'Расчётная мощность (цель):    {POWER*100:.0f}%')

Эмпирическая мощность должна оказаться в районе 78-82%, что подтверждает: наш расчёт верный, и при заявленных размерах групп мы действительно поймаем эффект 2 п.п. примерно в 4 случаях из 5.

## Что будет, если эффекта на самом деле нет: проверка на ложноположительные срабатывания

Симметричный тест: запустим симуляцию, где у обеих групп истинная конверсия одинаковая. Доля 'значимых' результатов должна быть примерно равна alpha = 5%. Если получится сильно больше, значит в дизайне теста что-то не так.

In [ ]:
false_positives = 0
for _ in range(N_SIMULATIONS):
    control_outcomes = rng.binomial(1, p_baseline, size=n_per_group)
    treat_outcomes = rng.binomial(1, p_baseline, size=n_per_group)
    n1, n2 = n_per_group, n_per_group
    p1 = control_outcomes.mean()
    p2 = treat_outcomes.mean()
    p_pool = (control_outcomes.sum() + treat_outcomes.sum()) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
    z = (p2 - p1) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    if p_value < ALPHA:
        false_positives += 1

fpr = false_positives / N_SIMULATIONS
print(f'Доля ложных срабатываний при отсутствии эффекта: {fpr*100:.1f}%')
print(f'Целевой alpha: {ALPHA*100:.0f}%')

Если получилось около 5%, всё работает корректно: тест честно держит обещание не выдавать ложноположительные результаты чаще, чем в одном случае из двадцати.

## Итог по дизайну

Что я зафиксировал заранее, до запуска теста:
- группы делим случайно, на момент совершения первой покупки в сегменте 'низкий чек';
- основная метрика: доля клиентов, совершивших вторую покупку в течение 30 дней;
- alpha = 5%, мощность 80%, MDE = 2 п.п. абсолютных;
- размер каждой группы посчитан выше;
- тест останавливаем по достижении нужных размеров групп, не раньше (не подсматриваем).

Дополнительно проверяем guardrail-метрики: средний чек второй покупки и общую выручку с клиента в течение 60 дней. Если триггерная кампания подняла retention, но за счёт каннибализации среднего чека, рост может оказаться нулевым в деньгах.